# Taste and smell clustering

Sarthak Aggarwal (MT26139) and Divyanshu Joshi (MT26120), CGAS Group 26, Deliverable 1

**Question.** Do the taste and smell tags carry real information? If they do, ingredients with similar tags should end up in the same cluster, and those clusters should line up with food categories (spices with spices, vegetables with vegetables).

**Method.** Each ingredient becomes a 0/1 vector: one position per taste label and per smell label. We cluster the vectors with k-means (written with numpy, no extra library) and compare the clusters to the categories in `D1_0_ingredient_categories.csv`.

**Honest check.** A purity score alone means little, so we also shuffle the categories randomly and compute purity again. If the real score is clearly higher than the shuffled one, the tags carry signal.

Files needed in the same folder: `D1_0_ingredient_250_final_dataset.csv`, `D1_0_ingredient_categories.csv`, `D1_0_tag_vocabulary.csv`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

data = pd.read_csv("D1_0_ingredient_250_final_dataset.csv", encoding="utf-8-sig")
cats = pd.read_csv("D1_0_ingredient_categories.csv", encoding="utf-8-sig")
vocab = pd.read_csv("D1_0_tag_vocabulary.csv", encoding="utf-8-sig")
vocab = vocab[vocab["label"] != "not available"]

df = data.merge(cats, on="english_name")
print(len(df), "ingredients,", df["category"].nunique(), "categories")
df["category"].value_counts()

## Step 1. Tags to vectors
One column per label. A 1 means the ingredient has that label. Ingredients with no taste and no smell at all (water) have an all-zero vector, so they are left out of the clustering.

In [ ]:
columns = ["taste:" + l for l in vocab[vocab["type"] == "gustatory"]["label"]] + \
          ["smell:" + l for l in vocab[vocab["type"] == "olfactory"]["label"]]
position = {name: i for i, name in enumerate(columns)}

X = np.zeros((len(df), len(columns)))
for row, (taste, smell) in enumerate(zip(df["gustatory"], df["olfactory"])):
    for label in taste.split("|"):
        if "taste:" + label in position:
            X[row, position["taste:" + label]] = 1
    for label in smell.split("|"):
        if "smell:" + label in position:
            X[row, position["smell:" + label]] = 1

has_tags = X.sum(axis=1) > 0
X, df = X[has_tags], df[has_tags].reset_index(drop=True)
y = df["category"].values
print("vector size:", X.shape[1], "| ingredients clustered:", len(df))

## Step 2. k-means
Plain k-means with the k-means++ way of choosing the start points, repeated 10 times; we keep the run with the smallest total distance.

In [ ]:
def kmeans(X, k, seed=0, n_init=10, iters=100):
    best = None
    for s in range(n_init):
        rng = np.random.default_rng(seed + s)
        centers = [X[rng.integers(len(X))]]
        for _ in range(k - 1):
            d = np.min([((X - c) ** 2).sum(axis=1) for c in centers], axis=0)
            centers.append(X[rng.choice(len(X), p=d / d.sum())] if d.sum() > 0 else X[rng.integers(len(X))])
        centers = np.array(centers)
        for _ in range(iters):
            labels = np.argmin(((X[:, None, :] - centers[None]) ** 2).sum(axis=2), axis=1)
            new = np.array([X[labels == j].mean(axis=0) if (labels == j).any() else centers[j] for j in range(k)])
            if np.allclose(new, centers):
                break
            centers = new
        inertia = ((X - centers[labels]) ** 2).sum()
        if best is None or inertia < best[0]:
            best = (inertia, labels)
    return best[1]

def purity(labels, truth):
    table = pd.crosstab(labels, truth)
    return table.max(axis=1).sum() / table.values.sum()

## Step 3. Do the clusters match the categories?
Purity = for each cluster take its most common category, add those counts up and divide by all ingredients. The shuffled column does the same with the categories mixed up at random (average of 200 shuffles). That is what we would get by luck.

In [ ]:
rows = []
for k in (5, 8, 10, 13):
    labels = kmeans(X, k)
    rng = np.random.default_rng(1)
    shuffled = np.mean([purity(labels, rng.permutation(y)) for _ in range(200)])
    rows.append({"k": k, "purity": round(purity(labels, y), 3), "purity if categories shuffled": round(shuffled, 3)})
pd.DataFrame(rows)

## Step 4. The 13-cluster result
k = 13 because there are 13 categories. The table shows, for each cluster, how many ingredients of each category it holds.

In [ ]:
K = df["category"].nunique()
cluster = kmeans(X, K)
df["cluster"] = cluster
print("purity at k =", K, ":", round(purity(cluster, y), 3))
pd.crosstab(df["cluster"], df["category"])

In [ ]:
# the most common labels in each cluster, so a cluster can be read as "this is the green earthy group"
for j in range(K):
    members = X[cluster == j]
    top = np.argsort(-members.mean(axis=0))[:4]
    names = ", ".join(columns[i].split(":")[1] for i in top)
    print(f"cluster {j:>2} ({len(members):>2} ingredients): {names}")

## Step 5. Picture for the demo
The 42-dimensional vectors are squashed to 2 dimensions with PCA (done with numpy). Left: coloured by the real category. Right: coloured by the cluster found from the tags alone. If the two pictures look alike, the tags carry real signal.

In [ ]:
Xc = X - X.mean(axis=0)
_, _, vt = np.linalg.svd(Xc, full_matrices=False)
pts = Xc @ vt[:2].T

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, column, title in ((axes[0], "category", "Real category"), (axes[1], "cluster", "Cluster from tags")):
    for i, value in enumerate(sorted(df[column].unique(), key=str)):
        m = (df[column] == value).values
        jitter = np.random.default_rng(i).normal(0, 0.04, size=(m.sum(), 2))
        ax.scatter(*(pts[m] + jitter).T, s=28, label=str(value), alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel("PCA 1")
    ax.set_ylabel("PCA 2")
    ax.legend(fontsize=7, loc="best")
plt.tight_layout()
plt.savefig("D1_5_taste_smell_clusters.png", dpi=150)
plt.show()

## What to say about the result
* The clusters are clearly better than chance, so the tags carry real signal.
* They are not perfect. Meat and seafood, nuts and seeds, pulses, fruit and dairy each end up mostly in one or two clusters. Spices are the opposite: they are spread over several clusters, because different spices smell very different (woody, anise, camphor, pungent). Vegetables and grains also blur together, since they share the same smell labels (green, earthy, mild, cooked).
* This is expected. The tags were assigned from general knowledge and kept coarse on purpose, so that two ingredients can be compared in Deliverable 3.
* Clustering uses only the tags, never the ingredient names or categories. The categories are used only afterwards, for checking.